# LLM Zoomcamp 2026 - Homework 2: Vector Search

本 notebook 完成 Vector Search Homework 的六个问题：

1. 将查询转换为 384 维 embedding
2. 计算 cosine similarity
3. 手动实现向量搜索
4. 使用 `minsearch.VectorSearch`
5. 比较关键词搜索与向量搜索
6. 使用 Reciprocal Rank Fusion 实现 hybrid search

作业原文：[Homework: Vector Search](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/cohorts/2026/02-vector-search/homework.md)

## Setup

本作业使用课程提供的 ONNX `Embedder`，模型是 `Xenova/all-MiniLM-L6-v2`。

ONNX 版本不依赖 PyTorch 或 CUDA，但生成的 embedding 与课程中的 sentence-transformers 版本一致。下面的代码会在模型不存在时自动下载。

In [1]:
from pathlib import Path

import numpy as np
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index, VectorSearch

from download import download
from embedder import Embedder


MODEL_PATH = Path("models/Xenova/all-MiniLM-L6-v2")
if not (MODEL_PATH / "model.onnx").exists():
    download("Xenova/all-MiniLM-L6-v2")

embedder = Embedder(MODEL_PATH)
print("Embedder ready")

Embedder ready


## Q1. Embedding a query

查询：

> How does approximate nearest neighbor search work?

`Embedder.encode()` 返回一个经过归一化的 384 维向量。

In [2]:
query = "How does approximate nearest neighbor search work?"
v = embedder.encode(query)

print("Embedding shape:", v.shape)
print("Q1 answer - v[0]:", float(v[0]))

Embedding shape: (384,)
Q1 answer - v[0]: -0.020582036807885073


**答案：`-0.02`**

实际值约为 `-0.02058`。单个向量维度通常没有独立的语义解释；embedding 的价值来自整个向量在空间中的方向和距离。

## Load the lesson documents

与 Homework 1 一样，从固定 commit `8c1834d` 读取课程所有 `lessons/` Markdown 文件。固定 commit 能保证所有人使用同一批 72 个页面。

In [3]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

print("Number of documents:", len(documents))
documents[0]

Number of documents: 72


{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

## Q2. Cosine similarity

`Embedder` 默认返回 L2-normalized vectors。因此：

```text
cosine_similarity(a, b) = a dot b
```

不需要再除以两个向量的模长。

In [4]:
target_filename = "02-vector-search/lessons/07-sqlitesearch-vector.md"
target_document = next(
    document
    for document in documents
    if document["filename"] == target_filename
)

document_vector = embedder.encode(target_document["content"])
similarity = document_vector.dot(v)

print("Q2 answer - cosine similarity:", float(similarity))

Q2 answer - cosine similarity: 0.361070280302606


**答案：`0.37`**

实际值约为 `0.36107`。它表明查询和该页面在 embedding 空间中有较明显的语义相关性，但并非几乎相同。

## Q3. Chunking and search by hand

完整页面可能包含多个主题，使 embedding 被无关段落稀释。这里将页面切为：

- chunk size：`2000` 字符
- step：`1000` 字符
- overlap：`1000` 字符

然后一次批量生成所有 chunk embeddings，组成矩阵 `X`。

In [5]:
chunks = chunk_documents(documents, size=2000, step=1000)
X = embedder.encode_batch([chunk["content"] for chunk in chunks])

scores = X.dot(v)
best_index = int(np.argmax(scores))
best_chunk = chunks[best_index]

print("Number of chunks:", len(chunks))
print("Embedding matrix shape:", X.shape)
print("Highest score:", float(scores[best_index]))
print("Q3 answer - filename:", best_chunk["filename"])
print("Chunk start:", best_chunk["start"])

Number of chunks: 295
Embedding matrix shape: (295, 384)
Highest score: 0.648901732433228
Q3 answer - filename: 02-vector-search/lessons/07-sqlitesearch-vector.md
Chunk start: 1000


**答案：`02-vector-search/lessons/07-sqlitesearch-vector.md`**

`X` 的形状是 `(295, 384)`：

- 295 行对应 295 个 chunks
- 384 列对应 embedding 维度

`X.dot(v)` 同时计算查询与全部 chunks 的 cosine similarity，`argmax` 找出分数最高的位置。

## Q4. Vector search with minsearch

手动矩阵乘法适合学习原理。实际项目通常把向量和原始文档交给专门的搜索组件管理。

In [6]:
vector_index = VectorSearch(
    keyword_fields=["filename"],
    numeric_fields=["start"],
)
vector_index.fit(X, chunks)

query_q4 = "What metric do we use to evaluate a search engine?"
query_vector_q4 = embedder.encode(query_q4)
vector_results_q4 = vector_index.search(query_vector_q4, num_results=5)

print("Q4 answer - first filename:", vector_results_q4[0]["filename"])
print("\nTop 5 vector results:")
for position, result in enumerate(vector_results_q4, start=1):
    print(f"{position}. {result['filename']} (start={result['start']})")

Q4 answer - first filename: 04-evaluation/lessons/05-search-metrics.md

Top 5 vector results:
1. 04-evaluation/lessons/05-search-metrics.md (start=0)
2. 04-evaluation/lessons/01-intro.md (start=0)
3. 01-agentic-rag/lessons/05-search.md (start=0)
4. 04-evaluation/lessons/01-intro.md (start=2000)
5. 04-evaluation/lessons/15-next-steps.md (start=0)


**答案：`04-evaluation/lessons/05-search-metrics.md`**

向量搜索匹配的是含义。即使文档没有逐字重复查询，它仍能识别“evaluate a search engine”和“search metrics”之间的语义关系。

## Q5. Text search vs vector search

使用同一批 chunks 建立关键词索引：

- Vector search：匹配语义
- Text search：匹配词项

然后比较两个方法各自的 top 5。

In [7]:
text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
    numeric_fields=["start"],
).fit(chunks)

query_q5 = "How do I store vectors in PostgreSQL?"

vector_results_q5 = vector_index.search(
    embedder.encode(query_q5),
    num_results=5,
)
text_results_q5 = text_index.search(query_q5, num_results=5)

print("Vector search:")
for position, result in enumerate(vector_results_q5, start=1):
    print(f"{position}. {result['filename']} (start={result['start']})")

print("\nText search:")
for position, result in enumerate(text_results_q5, start=1):
    print(f"{position}. {result['filename']} (start={result['start']})")

vector_files = {result["filename"] for result in vector_results_q5}
text_files = {result["filename"] for result in text_results_q5}
difference = vector_files - text_files

print("\nQ5 answer - vector only:", difference)

Vector search:
1. 02-vector-search/lessons/08-pgvector.md (start=0)
2. 02-vector-search/lessons/08-pgvector.md (start=3000)
3. 03-orchestration/lessons/05-rag.md (start=2000)
4. 02-vector-search/lessons/08-pgvector.md (start=1000)
5. 02-vector-search/lessons/08-pgvector.md (start=2000)

Text search:
1. 02-vector-search/lessons/02-embeddings.md (start=4000)
2. 03-orchestration/lessons/05-rag.md (start=1000)
3. 02-vector-search/lessons/01-intro.md (start=0)
4. 03-orchestration/lessons/05-rag.md (start=0)
5. 02-vector-search/lessons/01-intro.md (start=1000)

Q5 answer - vector only: {'02-vector-search/lessons/08-pgvector.md'}


**答案：`02-vector-search/lessons/08-pgvector.md`**

查询使用完整词 `PostgreSQL`，而相关页面主题是 `pgvector`。向量模型能识别这种语义关联；关键词搜索更依赖字面上的词项重合。

## Q6. Hybrid search with Reciprocal Rank Fusion

Hybrid search 同时利用关键词搜索和向量搜索。两种搜索的原始分数尺度不同，不能直接相加，因此使用 RRF，只根据排名计算：

```text
RRF(d) = sum(1 / (k + rank(d)))
```

同一个 chunk 如果在两个列表中都排名靠前，就会累积更高的融合分数。

In [8]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]


query_q6 = "How do I give the model access to tools?"

vector_results_q6 = vector_index.search(
    embedder.encode(query_q6),
    num_results=5,
)
text_results_q6 = text_index.search(query_q6, num_results=5)
hybrid_results = rrf([vector_results_q6, text_results_q6])

print("Vector search:")
for position, result in enumerate(vector_results_q6, start=1):
    print(f"{position}. {result['filename']} (start={result['start']})")

print("\nText search:")
for position, result in enumerate(text_results_q6, start=1):
    print(f"{position}. {result['filename']} (start={result['start']})")

print("\nHybrid search:")
for position, result in enumerate(hybrid_results, start=1):
    print(f"{position}. {result['filename']} (start={result['start']})")

print("\nQ6 answer - first filename:", hybrid_results[0]["filename"])

Vector search:
1. 01-agentic-rag/lessons/01-intro.md (start=2000)
2. 04-evaluation/lessons/02-ground-truth.md (start=1000)
3. 01-agentic-rag/lessons/16-other-frameworks.md (start=0)
4. 01-agentic-rag/lessons/15-frameworks.md (start=2000)
5. 01-agentic-rag/lessons/13-function-calling.md (start=4000)

Text search:
1. 01-agentic-rag/lessons/14-agentic-loop.md (start=0)
2. 01-agentic-rag/lessons/13-function-calling.md (start=4000)
3. 01-agentic-rag/lessons/13-function-calling.md (start=5000)
4. 01-agentic-rag/lessons/13-function-calling.md (start=1000)
5. 04-evaluation/lessons/02-ground-truth.md (start=3000)

Hybrid search:
1. 01-agentic-rag/lessons/13-function-calling.md (start=4000)
2. 01-agentic-rag/lessons/01-intro.md (start=2000)
3. 01-agentic-rag/lessons/14-agentic-loop.md (start=0)
4. 04-evaluation/lessons/02-ground-truth.md (start=1000)
5. 01-agentic-rag/lessons/16-other-frameworks.md (start=0)

Q6 answer - first filename: 01-agentic-rag/lessons/13-function-calling.md


**答案：`01-agentic-rag/lessons/13-function-calling.md`**

它在单独的 vector search 中排第 5，在 text search 中排第 2。由于同时出现在两个排名列表中，RRF 将两边的贡献相加，使它成为 hybrid search 的第一名。

## 总结

- Embedding 将语义映射到向量空间。
- 归一化向量的 dot product 等于 cosine similarity。
- Chunking 可以减少一个 embedding 中混入的无关主题。
- Keyword search 擅长精确词项，vector search 擅长语义和同义表达。
- Hybrid search 使用 RRF 合并互补的检索信号。
- 实际项目应通过 evaluation 数据集测量效果，而不是先验地认定某一种搜索一定最好。